In [137]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [138]:
from matplotlib import pyplot as plt
import numpy as np
from sklearn.linear_model import SGDRegressor
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, r2_score
import pandas as pd
from pandas.api.types import is_numeric_dtype as is_num

chunk_size = 50000
all_X_train = []
all_y_train = []
total_X_test = pd.DataFrame()
total_y_test = pd.Series(dtype='float64')

all_columns = ['ID', 'vendorid', 'tpep_pickup_datetime', 'tpep_dropoff_datetime',
       'passenger_count', 'trip_distance', 'ratecodeid', 'store_and_fwd_flag',
       'pulocationid', 'dolocationid', 'payment_type', 'fare_amount', 'extra',
       'mta_tax', 'tip_amount', 'tolls_amount', 'improvement_surcharge',
       'total_amount', 'congestion_surcharge', 'airport_fee', 'duration']

unneeded_columns = [ 
    'airport_fee', 'payment_type', 'congestion_surcharge', 'passenger_count',
    'vendorid', 'improvement_surcharge', 'tolls_amount', 'extra', 'tip_amount',
    'ratecodeid', 'store_and_fwd_flag'
]

needed_columns = list(set(all_columns) - set(unneeded_columns))
chunks = pd.read_csv("training_dataset.csv", chunksize=chunk_size, usecols=needed_columns)

models = {
    "RandomForestRegressor": RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1, warm_start=True),
    #"SGDRegressor": SGDRegressor(alpha=0.0001, eta0=0.0001, learning_rate="adaptive", warm_start=True)
}


In [139]:
from sklearn.neighbors import KNeighborsRegressor
from sklearn.model_selection import GridSearchCV
sample_size = 1500  # Define how many samples to pick per chunk

samples = pd.DataFrame(columns=['trip_distance', 'duration', 'fare_amount', 'dolocationid', 'mta_tax', 'pulocationid', 'total_amount'])

counter = 0
for df in chunks:
    counter += 1
    if counter > 650: break
    df.drop(columns=['ID'], inplace=True)
    df.dropna(inplace=True)
    
    # Fix times
    df['tpep_pickup_datetime'] = pd.to_datetime(df['tpep_pickup_datetime'])
    df['tpep_pickup_hour'] = df['tpep_pickup_datetime'].dt.hour
    df.drop(columns=['tpep_pickup_datetime', 'tpep_dropoff_datetime'], inplace=True)
    
    # Assert that all columns are numeric
    for col in df.columns:
        assert is_num(df[col]), f"The '{col}' column contained categorical values"
    
    df = df[(df['duration'] < 2500) & (df['duration'] > 60)]
    df = df[(df['trip_distance'] < 200) & (df['trip_distance'] > 0.25)]
    df = df[(df['fare_amount'] < 300) & (df['fare_amount'] > 0)]
    
    
    
    # Remove negative, zero and infinite values
    df = df[df.ge(0).all(axis=1)]
    df = df[df.le(np.inf).all(axis=1)]
   
    for col in df.columns:
        assert (df[col] >= 0).all(), f"The '{col}' column contained zero or negative values"
    for i in range(4):
        sample = df.sample(n=min(sample_size, len(df)), random_state=42)  # Take random sample from chunk
        samples = pd.concat([samples, sample]) 
    
        

X_data = samples.drop(columns=['duration'])
y_data = samples['duration']

X_train, X_test, y_train, y_test = train_test_split(X_data, y_data, test_size=0.2, random_state=42)

X_scaler = StandardScaler()
X_train = pd.DataFrame(X_scaler.fit_transform(X_train), columns=X_train.columns, index=X_train.index)
X_test = pd.DataFrame(X_scaler.transform(X_test), columns=X_test.columns, index=X_test.index)

y_train = np.log1p(y_train)
    #y_train = pd.Series(y_scaler.fit_transform(y_train.values.reshape(-1, 1)).flatten(), index=y_train.index)
y_test = np.log1p(y_test)

KNeighbor = KNeighborsRegressor(n_neighbors=7, weights='uniform', algorithm='auto', leaf_size=30, p=2, metric='minkowski', metric_params=None, n_jobs=None)



KNeighbor.fit(X_train, y_train)
KN_pred =  KNeighbor.predict(X_test)
    
    
mse = mean_squared_error(y_test, KN_pred)
r2 = r2_score(y_test, KN_pred)
print(f"MSE: {mse:.4f}")
print(f"R2 Score: {r2:.4f}")

/var/folders/bh/1_tst5bs5ql7mt66t103s6xw0000gn/T/ipykernel_10198/276985455.py:37: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  samples = pd.concat([samples, sample])


MSE: 0.0151
R2 Score: 0.9632


Correlation Matrix

In [140]:
# Load the CSV file
df_eval = pd.read_csv("evaluation_dataset.csv")

# Store IDs for final output
eval_ids = df_eval["ID"]

# Drop ID column
df_eval.drop(columns=["ID"], inplace=True)

# Convert datetime column
df_eval['tpep_pickup_datetime'] = pd.to_datetime(df_eval['tpep_pickup_datetime'])
df_eval['tpep_pickup_hour'] = df_eval['tpep_pickup_datetime'].dt.hour

cols_to_drop = unneeded_columns + ['tpep_pickup_datetime']
# Drop only existing columns
df_eval.drop(columns=[col for col in cols_to_drop if col in df_eval.columns], inplace=True)

# Standardize features
#X_scaler = RobustScaler(quantile_range=(1.0, 99.0))
eval_scaler = StandardScaler()
df_eval = pd.DataFrame(eval_scaler.fit_transform(df_eval), columns=df_eval.columns, index=df_eval.index)
print(df_eval.columns)
df_eval = df_eval[X_train.columns]
# Make predictions
y_pred_eval = KNeighbor.predict(df_eval)

# Reverse scaling
y_pred_real = np.expm1(y_pred_eval)  # Inverse transform
y_pred_real = y_pred_real.flatten()

# Save output
df_out = pd.DataFrame({"ID": eval_ids, "duration": y_pred_real}).set_index("ID")
print(df_out.head())
df_out.to_csv("submission.csv")

Index(['trip_distance', 'pulocationid', 'dolocationid', 'fare_amount',
       'mta_tax', 'total_amount', 'tpep_pickup_hour'],
      dtype='object')
                                         duration
ID                                               
e624ebae-2d2f-41c1-b30f-ca222e1137f0  1730.382775
b8ee7f2b-74da-4c5a-b959-eb93c8f18ce5   453.825018
65c77a1f-7c83-4044-99fb-5d61fed94ca9   170.766697
9b353c17-61a5-4bbd-a44c-ecbb381f62cf   679.623612
b61489e1-e755-4ed9-9550-a8e5086eea2b   775.241052
